# Phase 1 Input Noise Debug

Walk-through of the Phase 1 diagnostics for the PYNQ-Z2 audio input chain.
No FPGA fabric is rebuilt; everything runs from Python on the existing
block design.

Sections (run top-to-bottom; each can be re-run independently):

1. Codec register baseline
2. Output-side rule-out (digital zero, then 1 kHz sine)
3. Input capture under three conditions (shorted, silent source, guitar)
4. Side-by-side stats comparison
5. FFT plots
6. ADC HPF (R19[5]) on/off comparison
7. Decision table

Prerequisites:
- Headphones connected for the output tests if you want to hear them.
- Decide what is connected to the input *before* each capture cell.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from audio_lab_pynq.AudioLabOverlay import AudioLabOverlay
from audio_lab_pynq import diagnostics

OUTPUT_DIR = '/home/xilinx/audio_diag'
SAMPLE_RATE_HZ = 48000
NUM_FRAMES = SAMPLE_RATE_HZ  # 1 second per capture

os.makedirs(OUTPUT_DIR, exist_ok=True)
ovl = AudioLabOverlay()
print('overlay loaded; output dir =', OUTPUT_DIR)

## 1. Codec register baseline

Snapshot the ADAU1761 right after `config_codec()`. We diff against this
snapshot when toggling the ADC HPF later.

Things to look at:
- `R19_ADC_CONTROL` low nibble = 0x3 means both ADCs enabled; bit 5 is the digital HPF (DC blocker).
- `R20`/`R21` = 0x00 means no input attenuation (codec default).
- `R29`-`R32` are the output volumes; 0xE7 is the codec default.

In [ ]:
baseline_regs = ovl.codec.dump_registers()
print(ovl.codec.format_register_snapshot(baseline_regs))
print()
print('ADC HPF state (R19[5]):', ovl.codec.get_adc_hpf_state())
print('Input digital volume (R20, R21):', ovl.codec.get_input_digital_volume())

## 2. Output-side rule-out

Play digital zero, then a clean -18 dBFS sine. If either is dirty, the
noise is downstream of the FPGA and changing the input will not help.

| Result                      | Likely culprit                                  |
|-----------------------------|-------------------------------------------------|
| Zero noisy                  | DAC / HP amp / GND / R29-R32 / R35              |
| Sine noisy too              | same area; ground loop or supply-coupled hum    |
| Both clean, input still bad | input is the issue, continue below              |

In [ ]:
print('Playing 2 s of digital silence...')
ovl.output_zero_test(duration_s=2.0)
print('done')

In [ ]:
print('Playing 2 s of 1 kHz sine at -18 dBFS...')
ovl.output_sine_test(freq_hz=1000.0, duration_s=2.0, amplitude_dbfs=-18.0)
print('done')

## 3. Input capture: 3 conditions

Each cell captures one second of stereo line-in. Set up the input *before*
running each cell. The capture also writes `<label>.npy` and `<label>.csv`
under `OUTPUT_DIR`.

In [ ]:
input('>> SHORT or disconnect the input jack, then press Enter...')
shorted_samples, shorted_stats = ovl.diagnostic_capture(
    'shorted', num_frames=NUM_FRAMES, save_dir=OUTPUT_DIR)

In [ ]:
input('>> Connect a known-silent source (powered-off device), then press Enter...')
silent_samples, silent_stats = ovl.diagnostic_capture(
    'silence_source', num_frames=NUM_FRAMES, save_dir=OUTPUT_DIR)

In [ ]:
input('>> Connect the guitar; do NOT play (silent strings). Then press Enter...')
guitar_samples, guitar_stats = ovl.diagnostic_capture(
    'guitar', num_frames=NUM_FRAMES, save_dir=OUTPUT_DIR)

## 4. Side-by-side stats

Same numbers as printed during capture, in one table.

What to look for:
- `mean` >> 0 -> ADC DC offset; the ADC HPF (section 6) should fix it.
- `rms(shorted)` close to `rms(silence)` -> input stage self-noise dominates.
- `rms(guitar)` >> `rms(silence)` -> noise rides on top of guitar pickup wiring.
- `min`/`max` near +/-8388608 -> clipping; attenuate via `set_input_digital_volume`.

In [ ]:
def _row(label, ch, s):
    pdb = '-inf' if s['peak_dBFS'] == float('-inf') else '{:.2f}'.format(s['peak_dBFS'])
    return '{:<16} {:<6} {:>10.1f} {:>10.1f} {:>10.1f} {:>12d} {:>12d} {:>10}'.format(
        label, ch, s['mean'], s['rms'], s['peak_abs'], s['min'], s['max'], pdb)

header = '{:<16} {:<6} {:>10} {:>10} {:>10} {:>12} {:>12} {:>10}'.format(
    'condition', 'ch', 'mean', 'rms', 'peak_abs', 'min', 'max', 'peak_dBFS')
print(header)
print('-' * len(header))
for label, stats in [('shorted', shorted_stats),
                     ('silence', silent_stats),
                     ('guitar',  guitar_stats)]:
    for ch in ('left', 'right'):
        print(_row(label, ch, stats[ch]))

## 5. FFT plots

Plot left-channel spectrum for each condition. Look for:
- Narrow peaks at 50/60 Hz and harmonics -> mains hum (ground loop, shielding).
- A broad rise at very low frequency -> DC drift; ADC HPF should help.
- Peaks above ~10 kHz that vary with USB / SD / Wi-Fi activity -> switcher noise.
- Flat noise floor far below -100 dBFS -> codec is clean.

In [ ]:
def plot_fft(samples, title, ax):
    x = samples[:, 0].astype(np.float64) / diagnostics.FULL_SCALE_24
    n = x.size
    win = np.hanning(n)
    spec = np.abs(np.fft.rfft(x * win)) / (np.sum(win) / 2.0)
    freqs = np.fft.rfftfreq(n, d=1.0 / SAMPLE_RATE_HZ)
    spec_db = 20.0 * np.log10(np.maximum(spec, 1e-12))
    ax.semilogx(freqs[1:], spec_db[1:])
    ax.set_title(title)
    ax.set_xlabel('Hz')
    ax.set_ylabel('dBFS')
    ax.set_xlim(20, SAMPLE_RATE_HZ / 2)
    ax.set_ylim(-140, 0)
    ax.grid(True, which='both', linewidth=0.3)

fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
plot_fft(shorted_samples, 'shorted (L)', axes[0])
plot_fft(silent_samples, 'silence_source (L)', axes[1])
plot_fft(guitar_samples, 'guitar (L)', axes[2])
plt.tight_layout()

## 6. ADC HPF (R19[5]) on/off

Toggle the ADAU1761 digital HPF and re-measure with the input shorted.
This is *only* a DC blocker (~2 Hz corner at Fs = 48 kHz). It does not
remove 50/60 Hz hum. If `mean` is the dominant problem, this fixes it
without rebuilding the bitstream.

In [ ]:
before_r19 = ovl.codec.dump_registers(['R19_ADC_CONTROL'])
ovl.codec.enable_adc_hpf()
print('R19 diff:')
ovl.codec_register_diff(before_r19, ['R19_ADC_CONTROL'])
print()
input('>> Keep input shorted/disconnected, then press Enter to recapture...')
hpf_samples, hpf_stats = ovl.diagnostic_capture(
    'shorted_hpf_on', num_frames=NUM_FRAMES, save_dir=OUTPUT_DIR)
print()
for ch in ('left', 'right'):
    b = shorted_stats[ch]
    a = hpf_stats[ch]
    print('{:<6} mean: {:>8.1f} -> {:>8.1f}   rms: {:>8.1f} -> {:>8.1f}'.format(
        ch, b['mean'], a['mean'], b['rms'], a['rms']))

In [ ]:
# Optional: roll back the ADC HPF if you want to re-test other conditions
# without it. Comment out otherwise.
#
# ovl.codec.disable_adc_hpf()
# print('ADC HPF state:', ovl.codec.get_adc_hpf_state())

## 7. Decision table

Use the numbers above + this table to decide what to do next.

In [ ]:
diagnostics.print_decision_table()

## Saved artifacts

All captures land under `OUTPUT_DIR` as both `.npy` and `.csv`.
Useful follow-ups:
- Reload with `np.load('/home/xilinx/audio_diag/guitar.npy')` for offline analysis.
- Compare `shorted` vs `shorted_hpf_on` to confirm the HPF actually moved `mean` toward zero.
- If the mains-hum peaks remain after section 6, that is the strongest signal that Phase 2 (Clash-side input conditioner) is needed.

In [ ]:
import glob
for path in sorted(glob.glob(os.path.join(OUTPUT_DIR, '*'))):
    print(path, os.path.getsize(path), 'bytes')